# Step 5 — LLM Field Extraction

Two-stage extraction for each person:

1. **Identity verification** — The LLM assesses each source passage: is this truly about our target person (a newspaper editor/owner active in the right time period and states)? It considers name commonality, time-period match, and biographical consistency. Only passages judged `high` or `medium` confidence proceed.

2. **Field-level extraction** — From verified passages, the LLM extracts verbatim quotes relevant to specific biographical fields. These quotes are stored in `field_extractions` for manual coding in Step 5b.

### Fields extracted
| Field | What to look for |
|---|---|
| political_affiliation | Party membership, endorsements, political positions |
| wealthy_family | Mentions of family wealth, prominent family, inheritance |
| birthplace | City/state/country of birth |
| ethnicity | National origin, ethnic background |
| religion | Church membership, religious affiliation |
| year_of_birth | Birth year or age at a known date |
| year_of_death | Death year, obituary date |
| education_level | Schools attended, degrees, self-taught |
| military_service | Regiment, rank, war service |
| public_office_sought | Ran for office, appointed to position |
| railroad_stockholder | Stock ownership in any railroad |
| railroad_board_member | Served on railroad board of directors |
| railroad_company_owner | Owned or controlled a railroad company |
| railroad_professional_services | Provided professional services to a railroad (lawyer, land agent, surveyor, engineer, contractor, lobbyist) |
| family_connection_railroad | Family members with railroad/industrial ties |
| other_corporate_directorships | Bank boards, insurance companies, other corps |
| real_estate | Land holdings, property, real estate investments |
| wealth_at_death | Estate value, probate, net worth |
| founded | Founded newspapers or other businesses |
| papers_owned_count | Number of papers owned/edited simultaneously |
| internal_labor_struggles | Strikes, labor disputes at their newspaper |

In [ ]:
import sys, json, time, os
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

sys.path.insert(0, str(Path('.')))
from db import get_connection, set_progress, pending_persons, store_field_extraction

load_dotenv('../.env')
from google import genai

conn = get_connection()
STEP = '5_field_extraction'
ASSEMBLED_DIR = Path('assembled')

# Create field_extractions table if it doesn't exist yet
conn.executescript("""
CREATE TABLE IF NOT EXISTS field_extractions (
    id               INTEGER PRIMARY KEY AUTOINCREMENT,
    research_id      INTEGER NOT NULL REFERENCES persons(research_id),
    field_name       TEXT NOT NULL,
    extracted_text   TEXT NOT NULL,
    source_ref       TEXT,
    confidence       TEXT DEFAULT 'confirmed',
    identity_confidence TEXT DEFAULT 'high',
    extraction_model TEXT,
    extracted_at     TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    coded_value      TEXT,
    coded_at         TIMESTAMP
);
""")

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash-lite")
MAX_CONCURRENT = int(os.getenv("GEMINI_MAX_CONCURRENT", "15"))

# Thread-safe rate limiter: 1000 RPM with safety margin
_rate_lock = threading.Lock()
_request_times = []

def _wait_for_rate_limit():
    """Block until we're under 950 requests in the last 60s."""
    while True:
        now = time.time()
        with _rate_lock:
            _request_times[:] = [t for t in _request_times if now - t < 60]
            if len(_request_times) < 950:
                _request_times.append(now)
                return
        time.sleep(0.1)

# Thread-safe DB lock (sqlite3 connections aren't thread-safe)
_db_lock = threading.Lock()

print(f"Using model: {MODEL}")
print(f"Max concurrent requests: {MAX_CONCURRENT}")

FIELDS = [
    "political_affiliation",
    "wealthy_family",
    "birthplace",
    "ethnicity",
    "religion",
    "year_of_birth",
    "year_of_death",
    "education_level",
    "military_service",
    "public_office_sought",
    "railroad_stockholder",
    "railroad_board_member",
    "railroad_company_owner",
    "railroad_professional_services",
    "family_connection_railroad",
    "other_corporate_directorships",
    "real_estate",
    "wealth_at_death",
    "founded",
    "papers_owned_count",
    "internal_labor_struggles",
]

## Extraction prompt

In [ ]:
EXTRACTION_PROMPT = """You are a research assistant helping code biographical data about historical American newspaper editors and owners (1860s–1900s).

## TARGET PERSON
- Name: {name}
- Active years: {years}
- Role: newspaper {role}
- States: {states}
- Newspapers: {newspapers}
{enrichment_note}

## TASK

You will receive source texts gathered by searching for this person's name. Many passages may be about DIFFERENT people with the same or similar name. Your job has two parts:

### Part 1: Identity verification
For each source passage, judge whether it is truly about the target person above — a newspaper editor/owner active in the stated time period and states. Consider:
- Does the passage mention journalism, newspapers, editing, or publishing in connection with this name?
- Is the time period consistent (1860s–1900s)?
- Is the geographic context consistent with the known states?
- How common is this name? (e.g., "W. Thompson" could be anyone; "Whitelaw Reid" is distinctive)
- Could this be a different person with the same name (e.g., a different profession, different era, different state)?

Only extract from passages where you have HIGH or MEDIUM confidence that the passage is about the target person. Skip passages that are likely about someone else.

### Part 2: Field extraction
From the verified passages, extract VERBATIM QUOTES (exact text from the source, even if OCR-garbled) that provide evidence for any of these biographical fields:

1. **political_affiliation** — party membership, political endorsements, campaign support, editorial political stance
2. **wealthy_family** — family wealth, prominent family background, inheritance, "old money"
3. **birthplace** — city, state, or country of birth
4. **ethnicity** — national origin, immigrant background, ethnic identity
5. **religion** — church membership, religious denomination, religious activity
6. **year_of_birth** — birth year, or age mentioned at a datable event
7. **year_of_death** — death year, obituary date
8. **education_level** — schools, colleges, degrees, apprenticeships, self-educated
9. **military_service** — military rank, regiment, wars served in
10. **public_office_sought** — ran for office, held public position, appointed to government role
11. **railroad_stockholder** — owned stock/shares in a railroad company
12. **railroad_board_member** — served as director on a railroad board
13. **railroad_company_owner** — owned or controlled a railroad company
14. **railroad_professional_services** — provided professional services to a railroad (e.g., lawyer, attorney, legal counsel, land agent, surveyor, engineer, contractor, lobbyist for a railroad company)
15. **family_connection_railroad** — family members (spouse, father, in-laws) with railroad or industrial ties
16. **other_corporate_directorships** — bank boards, insurance companies, manufacturing, other corporate roles
17. **real_estate** — land holdings, property ownership, real estate investments
18. **wealth_at_death** — estate value, probate records, net worth estimates
19. **founded** — founded newspapers, businesses, or organizations (name what was founded)
20. **papers_owned_count** — evidence of owning/editing multiple papers (name them)
21. **internal_labor_struggles** — strikes, labor disputes, union conflicts at their newspaper

## OUTPUT FORMAT

Return a JSON object with this structure:
```json
{{
  "extractions": [
    {{
      "field": "field_name",
      "quote": "exact verbatim text from the source passage",
      "source_ref": "source title or label from the [SOURCE: ...] tag",
      "confidence": "confirmed or uncertain",
      "identity_confidence": "high or medium"
    }}
  ]
}}
```

RULES:
- The "quote" MUST be copied verbatim from the source text. Do not paraphrase or summarize.
- Keep quotes focused — include just enough context to understand the claim (1-3 sentences).
- If the same field has evidence from multiple sources, create separate extraction objects for each.
- Set confidence="uncertain" if the OCR is garbled or the meaning is ambiguous.
- If you find NO relevant information in any verified passage, return {{"extractions": []}}.
- Return ONLY valid JSON.

## SOURCE TEXTS
{source_texts}
"""


def format_source_texts(text_blocks, max_blocks=40, max_chars_per_block=1500):
    """
    Format text blocks for the prompt. Blocks are pre-sorted by relevance_score
    from step 4 assembly.
    """
    sorted_blocks = sorted(
        text_blocks,
        key=lambda b: (-b.get("relevance_score", 0 if b.get("is_keyword_filtered") else -1),)
    )

    lines = []
    total_chars = 0
    max_total = 50000  # ~12k tokens — slightly more budget for broader field set

    for i, block in enumerate(sorted_blocks[:max_blocks]):
        if total_chars >= max_total:
            break

        source_label = block["source_title"] or block["source_url"] or f"Source {i+1}"
        if block["item_id"]:
            source_label += f" [{block['item_id']}]"
        if block["page"]:
            source_label += f" (p. {block['page']})"

        text = block["context"] or block["passage"]
        text = text[:max_chars_per_block]

        entry = f"[SOURCE: {source_label}]\n{text}\n"
        lines.append(entry)
        total_chars += len(entry)

    return "\n---\n".join(lines)

## Extraction function

In [3]:
def extract_for_person(research_id, client, model, dry_run=False):
    """
    Run field extraction for one person. Returns (research_id, list_of_extractions, error_string).
    Thread-safe: reads files only, API call is stateless, DB writes use lock.
    """
    doc_path = ASSEMBLED_DIR / f"{research_id}.json"
    if not doc_path.exists():
        return research_id, None, "assembled document not found"

    with open(doc_path, encoding="utf-8") as f:
        doc = json.load(f)

    if not doc["text_blocks"]:
        return research_id, [], "no text passages"

    # Build enrichment note
    enr = doc.get("enrichment", {})
    enrichment_note = ""
    if enr:
        parts = []
        if enr.get("birth_year"):
            parts.append(f"Born {enr['birth_year']}")
        if enr.get("death_year"):
            parts.append(f"died {enr['death_year']}")
        if enr.get("birth_place"):
            parts.append(f"in {enr['birth_place']}")
        if enr.get("wikidata_qid"):
            parts.append(f"Wikidata: {enr['wikidata_qid']}")
        enrichment_note = "ENRICHMENT: " + "; ".join(parts) if parts else ""

    source_texts = format_source_texts(doc["text_blocks"])
    years_str = f"{doc['first_active_year'] or '?'}\u2013{doc['last_active_year'] or '?'}"

    prompt = EXTRACTION_PROMPT.format(
        name=doc["canonical_name"],
        years=years_str,
        role="editor/owner",
        states=", ".join(doc["known_states"]) or "unknown",
        newspapers=", ".join(doc["known_newspapers"][:5]) or "unknown",
        enrichment_note=enrichment_note,
        source_texts=source_texts,
    )

    if dry_run:
        return research_id, f"DRY RUN: prompt length {len(prompt)} chars, {len(doc['text_blocks'])} blocks", None

    # Call Gemini with rate limiting
    for attempt in range(3):
        try:
            _wait_for_rate_limit()
            response = client.models.generate_content(model=model, contents=prompt)
            text = response.text.strip()

            # Strip markdown code blocks if present
            if text.startswith("```"):
                text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()

            result = json.loads(text)
            extractions = result.get("extractions", [])
            if not isinstance(extractions, list):
                extractions = []

            # Validate field names
            valid = []
            for e in extractions:
                if not isinstance(e, dict):
                    continue
                if e.get("field") not in FIELDS:
                    continue
                if not e.get("quote"):
                    continue
                valid.append(e)

            return research_id, valid, None

        except json.JSONDecodeError as e:
            if attempt < 2:
                time.sleep(5)
        except Exception as e:
            err_str = str(e).lower()
            if "429" in err_str or "rate" in err_str:
                wait_time = (attempt + 1) * 10
                time.sleep(wait_time)
                _wait_for_rate_limit()
            elif attempt < 2:
                time.sleep(10 * (attempt + 1))

    return research_id, None, "failed after 3 attempts"


def save_field_extractions(conn, research_id, extractions, model_name):
    """Store field extractions in DB. Returns count. Must be called with _db_lock held."""
    count = 0
    for e in extractions:
        store_field_extraction(
            conn, research_id,
            field_name=e["field"],
            extracted_text=e["quote"],
            source_ref=e.get("source_ref"),
            confidence=e.get("confidence", "confirmed"),
            identity_confidence=e.get("identity_confidence", "high"),
            extraction_model=model_name,
        )
        count += 1
    return count

## Run extraction loop

In [4]:
DRY_RUN = False   # set True to preview prompts without calling API
ONLY_WITH_TEXT = True  # skip persons with 0 text passages

todo = pending_persons(conn, STEP)
print(f"{len(todo)} persons pending for step '{STEP}'")

if ONLY_WITH_TEXT:
    has_text = {
        r["research_id"]
        for r in conn.execute(
            "SELECT DISTINCT research_id FROM texts_downloaded"
        ).fetchall()
    }
    todo_filtered = [r for r in todo if r in has_text]
    print(f"  {len(todo_filtered)} have text passages (ONLY_WITH_TEXT=True)")
    todo = todo_filtered

if DRY_RUN:
    print("\nDRY RUN MODE — no API calls, no DB writes")

568 persons pending for step '5_field_extraction'
  367 have text passages (ONLY_WITH_TEXT=True)


In [5]:
from tqdm import tqdm

total_extractions = 0
persons_with_hits = 0
errors = 0

if DRY_RUN:
    # Sequential dry run — just preview
    for research_id in todo[:10]:
        rid, results, error = extract_for_person(research_id, client, MODEL, dry_run=True)
        if isinstance(results, str):
            person_name = conn.execute(
                "SELECT canonical_name FROM persons WHERE research_id=?", (rid,)
            ).fetchone()["canonical_name"]
            print(f"  [{rid}] {person_name}: {results}")
else:
    # Concurrent extraction with rate limiting
    pbar = tqdm(total=len(todo), desc="Extracting")

    with ThreadPoolExecutor(max_workers=MAX_CONCURRENT) as executor:
        futures = {
            executor.submit(extract_for_person, rid, client, MODEL): rid
            for rid in todo
        }

        for future in as_completed(futures):
            research_id, results, error = future.result()

            if error and results is None:
                with _db_lock:
                    set_progress(conn, research_id, STEP, 'failed', error_msg=error)
                errors += 1
                pbar.update(1)
                continue

            if results is None:
                results = []

            with _db_lock:
                count = save_field_extractions(conn, research_id, results, MODEL)
                set_progress(conn, research_id, STEP, 'done', result_count=count)

            total_extractions += count
            if count > 0:
                persons_with_hits += 1
            pbar.update(1)

    pbar.close()

    print(f"\nExtraction complete:")
    print(f"  Persons processed: {len(todo)}")
    print(f"  Persons with ≥1 extraction: {persons_with_hits}")
    print(f"  Total field extractions: {total_extractions}")
    print(f"  Errors: {errors}")

Extracting: 100%|██████████| 367/367 [06:46<00:00,  1.11s/it]


Extraction complete:
  Persons processed: 367
  Persons with ≥1 extraction: 176
  Total field extractions: 1137
  Errors: 0


## Results review

In [6]:
print("=== Field Extraction Results ===\n")

# Summary counts
total = conn.execute("SELECT COUNT(*) as n FROM field_extractions").fetchone()["n"]
persons_hit = conn.execute("SELECT COUNT(DISTINCT research_id) as n FROM field_extractions").fetchone()["n"]
print(f"Total field extractions: {total}")
print(f"Persons with ≥1 extraction: {persons_hit}\n")

# By field
print("Extractions by field:")
by_field = conn.execute("""
    SELECT field_name, COUNT(*) as n FROM field_extractions
    GROUP BY field_name ORDER BY n DESC
""").fetchall()
for r in by_field:
    print(f"  {r['field_name']:<40} {r['n']:>4}")

print()

# By confidence
print("By identity confidence:")
by_conf = conn.execute("""
    SELECT identity_confidence, COUNT(*) as n FROM field_extractions
    GROUP BY identity_confidence ORDER BY n DESC
""").fetchall()
for r in by_conf:
    print(f"  {r['identity_confidence']:<15} {r['n']:>4}")

print()

# Sample extractions
print("Sample extractions:")
print("-" * 80)
samples = conn.execute("""
    SELECT p.canonical_name, fe.field_name, fe.extracted_text, fe.source_ref, fe.confidence
    FROM field_extractions fe
    JOIN persons p ON fe.research_id = p.research_id
    ORDER BY RANDOM() LIMIT 10
""").fetchall()
for r in samples:
    print(f"  {r['canonical_name']} — {r['field_name']}")
    print(f"  [{r['confidence']}] {r['extracted_text'][:200]}")
    print(f"  Source: {r['source_ref']}")
    print()

=== Field Extraction Results ===

Total field extractions: 1171
Persons with ≥1 extraction: 178

Extractions by field:
  papers_owned_count                        240
  public_office_sought                      204
  political_affiliation                     140
  other_corporate_directorships              84
  founded                                    84
  year_of_death                              69
  education_level                            64
  military_service                           52
  real_estate                                48
  religion                                   42
  year_of_birth                              36
  birthplace                                 34
  railroad_board_member                      22
  wealthy_family                             16
  wealth_at_death                            13
  ethnicity                                  13
  railroad_company_owner                      5
  internal_labor_struggles                    2
  family_connecti

## Next step

Run **Step 5b** (`step5b_manual_coding.ipynb`) to manually code values from the extracted text using the interactive widget.